
# Probabilistic Delegations by Utility & Stickiness

This notebook reconstructs **delegations per epoch** using the probabilistic rule from the model:

- For each delegator \(i\) with stickiness \(s_i\):
  1. **Reconsider** with probability \(1 - s_i\).
  2. If there exists a strictly better DRep \(j^*\) (i.e., \(\Delta u = u_{ij^*,t} - u_{ij,t} > 0\)), **switch** to \(j^*\) with probability \(\Delta u\).
  3. Otherwise, **stay** with current DRep.

Where \(u_{ij,t} = 1 - |O_t(i) - O_t(j)|\in[0,1]\).

**Inputs (from `csv_out/`):**
- `dreps_state.csv` — columns: `epoch, drep_id, opinion, stake`
- `delegators_state.csv` — columns: `epoch, delegator_id, opinion, stake, s`

**Outputs (to `csv_out_probabilistic/`):**
1. `delegations_probabilistic.csv` — per-epoch mapping with diagnostics:  
   `epoch, delegator_id, drep_id, switched, delta_u, p_reconsider, p_switch_cond, p_overall`
2. `dreps_wprime_probabilistic.csv` — DRep weights per epoch:  
   `epoch, drep_id, opinion, stake, delegated_stake, Wprime`

Notes:
- Epoch 0 initialization: assign each delegator to the **closest** DRep by opinion.
- Randomness is controlled by a seed; set `SEED` below for reproducibility.


In [45]:

import pandas as pd
from pathlib import Path
import random

IN_DIR = Path("csv_out_exp")
OUT_DIR = Path("csv_out_probabilistic")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 12345
rng = random.Random(SEED)

dreps = pd.read_csv(IN_DIR / "dreps_state.csv")
deleg = pd.read_csv(IN_DIR / "delegators_state.csv")

# Basic checks
assert set(['epoch','drep_id','opinion','stake']).issubset(dreps.columns)
assert set(['epoch','delegator_id','opinion','stake','s']).issubset(deleg.columns)

epochs = sorted(dreps['epoch'].unique())
print("Epochs:", epochs[:5], "... total:", len(epochs))


Epochs: [0, 1, 2, 3, 4] ... total: 10


In [46]:

def utility(oi: float, oj: float) -> float:
    return 1.0 - abs(float(oi) - float(oj))

def closest_drep(opinion_i: float, D: pd.DataFrame) -> str:
    tmp = D[['drep_id','opinion']].copy()
    tmp['dist'] = (float(opinion_i) - tmp['opinion'].astype(float)).abs()
    tmp.sort_values(['dist','drep_id'], inplace=True)
    return tmp.iloc[0]['drep_id']


In [47]:

deleg_rows = []
wprime_rows = []

current_map = {}

for epoch in epochs:
    D = dreps.loc[dreps['epoch'] == epoch, ['drep_id','opinion','stake']].copy()
    A = deleg.loc[deleg['epoch'] == epoch, ['delegator_id','opinion','stake','s']].copy()

    # Initialize current (closest) for new delegators
    for _, row in A.iterrows():
        aid = row['delegator_id']
        if aid not in current_map:
            current_map[aid] = closest_drep(row['opinion'], D)

        drep_op = dict(zip(D['drep_id'], D['opinion']))
        drep_stake = dict(zip(D['drep_id'], D['stake']))

        # inside the for _ , row in A.iterrows(): loop
        aid = row['delegator_id']
        oi  = float(row['opinion'])
        si  = float(row['s'])
        st  = float(row['stake'])
        cur = current_map[aid]
        best = closest_drep(oi, D)

        u_cur = utility(oi, drep_op[cur])
        u_best = utility(oi, drep_op[best])
        delta_u = u_best - u_cur
        p_reconsider  = max(0.0, min(1.0, 1.0 - si))

        p_switch_cond = max(0.0, min(1.0, float(delta_u))) if delta_u > 0 else 0.0
        p_overall     = p_reconsider * p_switch_cond

        switched = 0
        if (rng.random() < p_reconsider) and (delta_u > 0) and (rng.random() < p_switch_cond):
            current_map[aid] = best
            switched = 1

        # 💡 include delegator opinion/stake/s and the matched drep opinion
        deleg_rows.append({
            'epoch': int(epoch),
            'delegator_id': aid,
            'opinion': oi,
            'stake': st,
            's': si,
            'drep_id': current_map[aid],
            'drep_opinion': float(drep_op[current_map[aid]]),
            'switched': int(switched),
            'delta_u': float(delta_u),
            'p_reconsider': float(p_reconsider),
            'p_switch_cond': float(p_switch_cond),
            'p_overall': float(p_overall),
        })


    # Compute Wprime for epoch
    Wprime = {d: float(st) for d, st in drep_stake.items()}
    for _, row in A.iterrows():
        aid = row['delegator_id']
        st  = float(row['stake'])
        dr  = current_map[aid]
        Wprime[dr] += st

    for d_id in D['drep_id']:
        wprime_rows.append({
            'epoch': int(epoch),
            'drep_id': d_id,
            'opinion': float(drep_op[d_id]),
            'stake': float(drep_stake[d_id]),
            'delegated_stake': float(Wprime[d_id] - drep_stake[d_id]),
            'Wprime': float(Wprime[d_id]),
        })

import pandas as pd
deleg_df = pd.DataFrame(deleg_rows, columns=[
    'epoch','delegator_id','opinion','stake','s',
    'drep_id','drep_opinion','switched','delta_u',
    'p_reconsider','p_switch_cond','p_overall'
])

wprime_df = pd.DataFrame(wprime_rows, columns=[
    'epoch','drep_id','opinion','stake','delegated_stake','Wprime'
])

out1 = OUT_DIR / "delegations_probabilistic.csv"
out2 = OUT_DIR / "dreps_wprime_probabilistic.csv"
deleg_df.to_csv(out1, index=False)
wprime_df.to_csv(out2, index=False)

print("Saved:", out1.resolve())
print("Saved:", out2.resolve())


Saved: /Users/Joel/Documents/GitHub/ada_drep/simulation/csv_out_probabilistic/delegations_probabilistic.csv
Saved: /Users/Joel/Documents/GitHub/ada_drep/simulation/csv_out_probabilistic/dreps_wprime_probabilistic.csv
